In [198]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE

import seaborn as sns

In [199]:
dataset_path = "/kaggle/input/datasets/sabahesaraki/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"

class_counts = {}

for category in os.listdir(dataset_path):

    category_path = os.path.join(dataset_path, category)

    if os.path.isdir(category_path):

        images = [img for img in os.listdir(category_path) if "mask" not in img]

        class_counts[category] = len(images)

print("Dataset Distribution")
for k,v in class_counts.items():
    print(f"{k} : {v}")

Dataset Distribution
benign : 437
normal : 133
malignant : 210


In [200]:
IMG_SIZE = 128

images = []
labels = []

classes = ["benign", "malignant", "normal"]

for label, category in enumerate(classes):

    folder_path = os.path.join(dataset_path, category)

    for img_name in os.listdir(folder_path):

        # Skip mask images
        if "mask" in img_name.lower():
            continue

        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = preprocess_input(img)

        images.append(img)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Dataset shape:", images.shape)
print("Labels shape:", labels.shape)

Dataset shape: (780, 128, 128, 3)
Labels shape: (780,)


In [201]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [202]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

for layer in base_model.layers[-30:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=base_model.input, outputs=x)

In [203]:
train_features = feature_extractor.predict(X_train, batch_size=32)
val_features = feature_extractor.predict(X_val, batch_size=32)
test_features = feature_extractor.predict(X_test, batch_size=32)

train_features = train_features.reshape(train_features.shape[0], -1)
val_features = val_features.reshape(val_features.shape[0], -1)
test_features = test_features.reshape(test_features.shape[0], -1)

print("Feature shape:", train_features.shape)

18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 249ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Feature shape: (546, 1280)


In [204]:
baseline_model = Sequential([

    Dense(256, activation='relu', input_shape=(train_features.shape[1],)),
    tf.keras.layers.BatchNormalization(),
    Dropout(0.5),

    Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),

    Dense(3, activation='softmax')

])

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_baseline = baseline_model.fit(
    train_features,
    y_train,
    validation_data=(val_features,y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 160ms/step - accuracy: 0.3804 - loss: 1.3459 - val_accuracy: 0.6325 - val_loss: 0.8070
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6320 - loss: 0.8291 - val_accuracy: 0.7265 - val_loss: 0.6501
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7148 - loss: 0.6504 - val_accuracy: 0.7863 - val_loss: 0.5983
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7556 - loss: 0.5506 - val_accuracy: 0.7949 - val_loss: 0.5576
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7779 - loss: 0.5098 - val_accuracy: 0.8120 - val_loss: 0.5004
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8221 - loss: 0.4376 - val_accuracy: 0.8462 - val_loss: 0.4628
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8543 - loss: 0.3733 - val_accuracy: 0.8632 - val_loss: 0.4516
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8230 - loss: 0.3979 - val_accuracy: 0.8718 - val_loss: 0.4208
E

In [205]:
y_pred_baseline = baseline_model.predict(test_features)
y_pred_baseline = np.argmax(y_pred_baseline, axis=1)

baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_prec = precision_score(y_test, y_pred_baseline, average="weighted")
baseline_rec = recall_score(y_test, y_pred_baseline, average="weighted")
baseline_f1 = f1_score(y_test, y_pred_baseline, average="weighted")

print("Baseline Accuracy:", baseline_acc)
print(classification_report(y_test, y_pred_baseline))

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step
Baseline Accuracy: 0.7435897435897436
              precision    recall  f1-score   support

           0       0.74      0.83      0.79        66
           1       0.77      0.55      0.64        31
           2       0.71      0.75      0.73        20

    accuracy                           0.74       117
   macro avg       0.74      0.71      0.72       117
weighted avg       0.75      0.74      0.74       117



In [206]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(train_features, y_train)

unique, counts = np.unique(y_train_smote, return_counts=True)

print("Class distribution after SMOTE")

for u,c in zip(unique,counts):
    print(f"Class {u}: {c}")

Class distribution after SMOTE
Class 0: 306
Class 1: 306
Class 2: 306


In [207]:
smote_model = Sequential([

    Dense(256, activation='relu', input_shape=(X_train_smote.shape[1],)),
    Dropout(0.5),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(64, activation='relu'),

    Dense(3, activation='softmax')
])

smote_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_smote = smote_model.fit(
    X_train_smote,
    y_train_smote,
    validation_data=(val_features,y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 86ms/step - accuracy: 0.4224 - loss: 1.2590 - val_accuracy: 0.6667 - val_loss: 0.7729
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6745 - loss: 0.7951 - val_accuracy: 0.7009 - val_loss: 0.6578
Epoch 3/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7114 - loss: 0.6422 - val_accuracy: 0.7436 - val_loss: 0.5842
Epoch 4/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7796 - loss: 0.5649 - val_accuracy: 0.8120 - val_loss: 0.5334
Epoch 5/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8195 - loss: 0.4723 - val_accuracy: 0.7949 - val_loss: 0.5387
Epoch 6/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8438 - loss: 0.3920 - val_accuracy: 0.7863 - val_loss: 0.5349
Epoch 7/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8690 - loss: 0.3439 - val_accuracy: 0.7778 - val_loss: 0.4873
Epoch 8/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8936 - loss: 0.2886 - val_accuracy: 0.8547 - val_loss: 0.4102
Ep

In [208]:
y_pred_smote = smote_model.predict(test_features)
y_pred_smote = np.argmax(y_pred_smote, axis=1)

smote_acc = accuracy_score(y_test, y_pred_smote)

print("SMOTE Accuracy:", smote_acc)
print(classification_report(y_test, y_pred_smote))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
SMOTE Accuracy: 0.7777777777777778
              precision    recall  f1-score   support

           0       0.81      0.85      0.83        66
           1       0.73      0.71      0.72        31
           2       0.72      0.65      0.68        20

    accuracy                           0.78       117
   macro avg       0.76      0.74      0.75       117
weighted avg       0.78      0.78      0.78       117



In [209]:
aug_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.15,
    horizontal_flip=True
)

In [210]:
augmented_images = []
augmented_labels = []

for i in range(len(X_train)):

    img = X_train[i]
    label = y_train[i]

    img = img.reshape((1,128,128,3))

    aug_iter = aug_datagen.flow(img, batch_size=1)

    aug_img = next(aug_iter)[0]

    augmented_images.append(aug_img)
    augmented_labels.append(label)

augmented_images = np.array(augmented_images)
augmented_labels = np.array(augmented_labels)

print("Augmented images:", augmented_images.shape)

Augmented images: (546, 128, 128, 3)


In [211]:
X_train_aug = np.concatenate((X_train, augmented_images))
y_train_aug = np.concatenate((y_train, augmented_labels))

print("Combined training data:", X_train_aug.shape)

Combined training data: (1092, 128, 128, 3)


In [212]:
train_features_aug = feature_extractor.predict(X_train_aug, batch_size=32)

train_features_aug = train_features_aug.reshape(
    train_features_aug.shape[0], -1
)

print("Augmented feature shape:", train_features_aug.shape)

35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step
Augmented feature shape: (1092, 1280)


In [213]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_aug),
    y=y_train_aug
)

class_weights = dict(enumerate(class_weights))

print("Class weights:", class_weights)

Class weights: {0: np.float64(0.5947712418300654), 1: np.float64(1.2380952380952381), 2: np.float64(1.956989247311828)}


In [219]:
aug_model = Sequential([

    tf.keras.Input(shape=(train_features_aug.shape[1],)),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(64, activation='relu'),

    Dense(3, activation='softmax')

])

aug_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_aug = aug_model.fit(
    train_features_aug,
    y_train_aug,
    validation_data=(val_features, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights
)

Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.3974 - loss: 1.3160 - val_accuracy: 0.7350 - val_loss: 0.7813
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5907 - loss: 0.8732 - val_accuracy: 0.7265 - val_loss: 0.6521
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6641 - loss: 0.7373 - val_accuracy: 0.6923 - val_loss: 0.6554
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6780 - loss: 0.7115 - val_accuracy: 0.7094 - val_loss: 0.5774
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7362 - loss: 0.6102 - val_accuracy: 0.8205 - val_loss: 0.5207
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7679 - loss: 0.5526 - val_accuracy: 0.7350 - val_loss: 0.5152
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7821 - loss: 0.5035 - val_accuracy: 0.7265 - val_loss: 0.6140
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7765 - loss: 0.4735 - val_accuracy: 0.7863 - val_loss

In [217]:
y_pred_aug = aug_model.predict(test_features)
y_pred_aug = np.argmax(y_pred_aug, axis=1)

aug_acc = accuracy_score(y_test, y_pred_aug)

print("Augmentation + Class Weight Accuracy:", aug_acc)
print(classification_report(y_test, y_pred_aug))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Augmentation + Class Weight Accuracy: 0.8034188034188035
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        66
           1       0.75      0.68      0.71        31
           2       0.76      0.80      0.78        20

    accuracy                           0.80       117
   macro avg       0.78      0.78      0.78       117
weighted avg       0.80      0.80      0.80       117



In [218]:
print("Baseline Accuracy:", baseline_acc)
print("SMOTE Accuracy:", smote_acc)
print("Augmentation + Class Weight Accuracy:", aug_acc)

Baseline Accuracy: 0.7435897435897436
SMOTE Accuracy: 0.7777777777777778
Augmentation + Class Weight Accuracy: 0.8034188034188035
